In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
address = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/Address")
address.limit(5).display()

AddressID,AddressLine1,AddressLine2,City,StateProvince,CountryRegion,PostalCode,rowguid,ModifiedDate,_rescued_data
9,8713 Yosemite Ct.,null,Bothell,Washington,United States,98011,268af621-76d7-4c78-9441-144fd139821a,2006-07-01 00:00:00.0000000,null
11,1318 Lasalle Street,null,Bothell,Washington,United States,98011,981b3303-aca2-49c7-9a96-fb670785b269,2007-04-01 00:00:00.0000000,null
25,9178 Jumping St.,null,Dallas,Texas,United States,75201,c8df3bd9-48f0-4654-a8dd-14a67a84d3c6,2006-09-01 00:00:00.0000000,null
28,9228 Via Del Sol,null,Phoenix,Arizona,United States,85004,12ae5ee1-fc3e-468b-9b92-3b970b169774,2005-09-01 00:00:00.0000000,null
32,26910 Indela Road,null,Montreal,Quebec,Canada,H1Y 2H5,84a95f62-3ae8-4e7e-bbd5-5a6f00cd982d,2006-08-01 00:00:00.0000000,null


In [0]:
address.columns

['AddressID',
 'AddressLine1',
 'AddressLine2',
 'City',
 'StateProvince',
 'CountryRegion',
 'PostalCode',
 'rowguid',
 'ModifiedDate',
 '_rescued_data']

In [0]:
address = address.withColumnRenamed('AddressID', 'Address_id')\
                    .withColumnRenamed('AddressLine1', 'Address_line1')\
                    .withColumnRenamed('AddressLine2', 'Address_line2')\
                    .withColumnRenamed('StateProvince', 'State_province')\
                    .withColumnRenamed('CountryRegion', 'Country_region')\
                    .withColumnRenamed('PostalCode', 'Postal_code')\
                    .withColumnRenamed('ModifiedDate', 'Modified_date')

address = address.drop("_rescued_data")

address = address.withColumn('Modified_date', to_date(col('Modified_date')))
address.limit(5).display()

Address_id,Address_line1,Address_line2,City,State_province,Country_region,Postal_code,rowguid,Modified_date
9,8713 Yosemite Ct.,null,Bothell,Washington,United States,98011,268af621-76d7-4c78-9441-144fd139821a,2006-07-01
11,1318 Lasalle Street,null,Bothell,Washington,United States,98011,981b3303-aca2-49c7-9a96-fb670785b269,2007-04-01
25,9178 Jumping St.,null,Dallas,Texas,United States,75201,c8df3bd9-48f0-4654-a8dd-14a67a84d3c6,2006-09-01
28,9228 Via Del Sol,null,Phoenix,Arizona,United States,85004,12ae5ee1-fc3e-468b-9b92-3b970b169774,2005-09-01
32,26910 Indela Road,null,Montreal,Quebec,Canada,H1Y 2H5,84a95f62-3ae8-4e7e-bbd5-5a6f00cd982d,2006-08-01


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricksintech.silver;

CREATE TABLE IF NOT EXISTS databricksintech.silver.address
(
    Address_id  INTEGER,
    Address_line1 STRING,
    Address_line2 STRING,
    City STRING,
    State_province STRING,
    Country_region STRING,
    Postal_code STRING,
    rowguid STRING,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/address';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.address")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = address.alias("source"),
    condition = (
        "target.Address_id = source.Address_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
